In [1]:
from unsloth_mlx import FastLanguageModel
import torch
from unsloth_mlx.chat_templates import get_chat_template
from datasets import load_dataset
from transformers import TextStreamer
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
from pprint import pprint

/Users/rahi/miniconda3/envs/ML/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
max_seq_length = 1024
dtype = None  # torch.float16, torch.bfloat16, or None for float32
load_in_4bit = True
model_name_or_path = "ryuma007/qwen3-0.6B-hishab_json"

In [3]:
# model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name = model_name_or_path,
#         max_seq_length = max_seq_length,
#         dtype = dtype,
#         load_in_4bit = load_in_4bit,
#     )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

model = AutoPeftModelForCausalLM.from_pretrained(
    "ryuma007/qwen3-0.6B-hishab_json",  # YOUR MODEL YOU USED FOR TRAINING
    # load_in_4bit=True,
).to("mps")
tokenizer = AutoTokenizer.from_pretrained("ryuma007/qwen3-0.6B-hishab_json")

print("Model and Tokenizer loaded successfully!")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model Data type: {next(model.parameters()).dtype}")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2009.19it/s, Materializing param=model.norm.weight]                              


Model and Tokenizer loaded successfully!
Model device: mps:0
Model Data type: torch.bfloat16


In [4]:
def apply_template(system_prompt, prompts):
    all_messages = []
    for prompt in prompts:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
        all_messages.append(messages)
    return all_messages


system_prompt = "You are a financial assistant. Extract transactions from text into a structured JSON list."

prompts = [
    # "Got my salary: 80000. Also bought a new camera for 25000.",
    # "Received partial refund for flight cancellation: 3000.",
    # "Got 1k as pocket money and bought a gpu for 450.2k",
    # "Got 500 tk as gift from uncle. Food Cost was 150 tk. Bought a book for 300 tk.",
    "আজকে দোকানে ১২শ টাকার জিনিস কিনলাম,  খাবার খরচ ১০০ টাকা, যাতায়াত ৫০ টাকা, আর বন্ধুর ধার শোধ করলাম ৩০০ টাকা।",
]

all_messages = apply_template(system_prompt, prompts)

In [ ]:
def input_template(tokenizer, messages, device="mps"):
    return tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # This tells the model: "It's your turn to speak now"
        return_tensors = "pt",
    ).to(device)

def generate_response(model, input_ids, max_new_tokens=200, temperature=0.2, device="mps"):

    output_ids = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        use_cache=True,
    ).to(device)

    decoded_output = tokenizer.batch_decode(output_ids[:, input_ids.shape[1]:], skip_special_tokens=True)

    return decoded_output

i = 1

for messages in all_messages:
    inputs = input_template(tokenizer, messages, device="mps")
    responses = generate_response(model, inputs.input_ids, max_new_tokens=200, temperature=0.2, device="mps")
    print(f"Response - {i}")
    print(f"Generated Response: {responses}")
    print("\n")
    i += 1
    



Response - 1
Generated Response: ['[{"description":"দোকান খরচ","amount":1200.0,"category":"grocery","type":"expense"}, {"description":"খাবার খরচ","amount":100.0,"category":"food","type":"expense"}, {"description":"যাতায়াত","amount":50.0,"category":"transport","type":"expense"}, {"description":"বন্ধুর ধার","amount":300.0,"category":"other","type":"expense"}]']




In [8]:
# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer)
# model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

outputs = model.generate(
    input_ids = inputs, 
    max_new_tokens = 256, 
    use_cache = True,
    temperature = 0.1, # Keep low for JSON consistency
)
print(outputs)

# decoded_output = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)

# print(decoded_output)

AttributeError: 